In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q lightgbm catboost scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

SEEDS = [42, 7, 123]
N_SPLITS = 10

In [3]:
# ── Cell 3: Load data ────────────────────────────────────────────────────────
TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

print(f'Train: {TRAIN_DATA.shape}, Test: {TEST_DATA.shape}')
print(TRAIN_LABEL['disorder'].value_counts().sort_index())

Train: (13249, 41), Test: (8834, 41)
disorder
0     389
1    2068
2    1090
3    3096
4      58
5    1700
6     813
7    2643
8      91
9    1301
Name: count, dtype: int64


In [4]:
# ── Cell 4: Preprocessing ────────────────────────────────────────────────────
# Strategy:
#   - a3's feature engineering (interaction features, symptom patterns, etc.)
#   - Run1's MISSING category for high-missing cols (birth_asphyxia, radiation, substance_abuse)
#   - Individual missing flags (from a3) for ~10% missing cols
#   - Numeric encoding (a3 style) — more stable than CatBoost native cats

def preprocess(df):
    df = df.copy()

    # ── Drop zero-signal columns ─────────────────────────────────────────────
    drop_cols = [
        'first_name', 'last_name', 'insitute_name', 'institute_location',
        'test_1', 'test_2', 'test_3', 'test_4', 'test_5',
        'treatment_consent',
        'autopsy',        # 70% missing, uniform across all classes
    ]
    df = df.drop(columns=drop_cols)

    # ── Missing flags BEFORE encoding (a3 approach) ──────────────────────────
    miss_cols = [
        'gender', 'maternal_defect', 'mother_age', 'father_age',
        'respiration', 'heart_rate', 'risk_level', 'place_birth',
        'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies', 'abortion_cnt',
        'birth_defects', 'white_blood_cell_count', 'blood_test',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    df['missing_count']      = df[miss_cols].isna().sum(axis=1)
    df['missing_parent_age'] = df['mother_age'].isna().astype(int) + df['father_age'].isna().astype(int)
    df['missing_symptoms']   = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].isna().sum(axis=1)
    df['missing_clinical']   = df[['respiration','heart_rate','risk_level','blood_test']].isna().sum(axis=1)

    # Individual missing flags for key cols
    for col in ['mother_age','father_age','maternal_defect','gender',
                'risk_level','heart_rate','respiration','abortion_cnt','white_blood_cell_count']:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    # ── High-missing cols: fill with 'MISSING' string BEFORE mapping ─────────
    # birth_asphyxia, radiation_exposure, substance_abuse have ~32% missing
    # Treating missing as its own category captures the signal that data is absent
    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].fillna('MISSING')

    # ── Encode binary Y/N cols ───────────────────────────────────────────────
    binary_yn = [
        'mother_defect','father_defect','maternal_defect','paternal_defect',
        'alive','folic_acid','maternal_illness','infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1','symptom_2','symptom_3','symptom_4','symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y': 1, 'N': 0})

    df['respiration']  = df['respiration'].map({'A': 1, 'N': 0})
    df['heart_rate']   = df['heart_rate'].map({'A': 1, 'N': 0})
    df['risk_level']   = df['risk_level'].map({'H': 1, 'L': 0})
    df['place_birth']  = df['place_birth'].map({'I': 1, 'H': 0})
    df['birth_defects']= df['birth_defects'].map({'S': 1, 'M': 2})
    df['gender']       = df['gender'].map({'M': 0, 'F': 1, 'A': 2})
    df['blood_test']   = df['blood_test'].map({'N': 0, 'I': 1, 'S': 2, 'A': 3})

    # High-missing 3-category cols: Y=1, N=0, NR=2, MISSING=3
    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].map({'Y': 1, 'N': 0, 'NR': 2, 'MISSING': 3})

    # ── Engineered features (a3) ─────────────────────────────────────────────
    df['defect_sum']   = df[['mother_defect','father_defect','maternal_defect','paternal_defect']].sum(axis=1)
    df['symptom_sum']  = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].sum(axis=1)

    df['defect_x_symptom']     = df['defect_sum'] * df['symptom_sum']
    df['any_defect']           = (df['defect_sum'] > 0).astype(int)
    df['any_symptom']          = (df['symptom_sum'] > 0).astype(int)
    df['high_symptom']         = (df['symptom_sum'] >= 4).astype(int)
    df['all_defects']          = (df['defect_sum'] == 4).astype(int)
    df['parent_age_gap']       = (df['father_age'] - df['mother_age']).abs()
    df['symptom_defect_ratio'] = df['symptom_sum'] / (df['defect_sum'] + 1)

    # Symptom pattern features (highly discriminative per EDA)
    df['s4_and_s5']    = ((df['symptom_4'] == 1) & (df['symptom_5'] == 1)).astype(int)
    df['no_s4_s5']     = ((df['symptom_4'] == 0) & (df['symptom_5'] == 0)).astype(int)
    df['late_vs_early']= (df['symptom_4'].fillna(0) + df['symptom_5'].fillna(0)
                         - df['symptom_1'].fillna(0) - df['symptom_2'].fillna(0))
    df['weighted_sym'] = (df['symptom_1'].fillna(0)*1 + df['symptom_2'].fillna(0)*1 +
                          df['symptom_3'].fillna(0)*1 + df['symptom_4'].fillna(0)*2 +
                          df['symptom_5'].fillna(0)*2)

    df['both_parents_defect'] = ((df['mother_defect'] == 1) & (df['father_defect'] == 1)).astype(int)
    df['no_parent_defect']    = ((df['mother_defect'] == 0) & (df['father_defect'] == 0)).astype(int)

    return df


X_train = preprocess(TRAIN_DATA)
X_test  = preprocess(TEST_DATA)
y_train = TRAIN_LABEL['disorder'].values

print(f'X_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')
print(f'Features: {list(X_train.columns)}')

X_train shape: (13249, 58)
X_test  shape: (8834, 58)
Features: ['age', 'gender', 'mother_defect', 'father_defect', 'maternal_defect', 'paternal_defect', 'blood_cell_count', 'mother_age', 'father_age', 'alive', 'respiration', 'heart_rate', 'risk_level', 'birth_asphyxia', 'place_birth', 'folic_acid', 'maternal_illness', 'radiation_exposure', 'substance_abuse', 'infertility_treatment', 'problem_previous_pregnancies', 'abortion_cnt', 'birth_defects', 'white_blood_cell_count', 'blood_test', 'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5', 'missing_count', 'missing_parent_age', 'missing_symptoms', 'missing_clinical', 'mother_age_missing', 'father_age_missing', 'maternal_defect_missing', 'gender_missing', 'risk_level_missing', 'heart_rate_missing', 'respiration_missing', 'abortion_cnt_missing', 'white_blood_cell_count_missing', 'defect_sum', 'symptom_sum', 'defect_x_symptom', 'any_defect', 'any_symptom', 'high_symptom', 'all_defects', 'parent_age_gap', 'symptom_defect_ratio', 

In [5]:
# ── Cell 5: Class weights ────────────────────────────────────────────────────
class_counts  = np.bincount(y_train)
class_weights = len(y_train) / (10 * class_counts)   # same formula as a3

print('Class weights:')
for i, (n, w) in enumerate(zip(class_counts, class_weights)):
    print(f'  Class {i}: weight={w:.3f}  (n={n})')

Class weights:
  Class 0: weight=3.406  (n=389)
  Class 1: weight=0.641  (n=2068)
  Class 2: weight=1.216  (n=1090)
  Class 3: weight=0.428  (n=3096)
  Class 4: weight=22.843  (n=58)
  Class 5: weight=0.779  (n=1700)
  Class 6: weight=1.630  (n=813)
  Class 7: weight=0.501  (n=2643)
  Class 8: weight=14.559  (n=91)
  Class 9: weight=1.018  (n=1301)


In [6]:
# ── Cell 6: Training — 3-seed averaging, 10-fold CV ─────────────────────────
all_oof_proba  = np.zeros((len(y_train), 10))
all_test_preds = np.zeros((len(X_test), 10))

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    oof_proba  = np.zeros((len(y_train), 10))
    test_preds = np.zeros((len(X_test), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr,  X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr,  y_val = y_train[tr_idx],      y_train[val_idx]

        model = CatBoostClassifier(
            iterations           = 2000,
            learning_rate        = 0.03,   # same as a3 — slower, safer
            depth                = 6,
            l2_leaf_reg          = 3,
            min_data_in_leaf     = 10,
            class_weights        = class_weights,
            early_stopping_rounds= 100,
            eval_metric          = 'Accuracy',
            random_seed          = SEED,
            verbose              = 0,
            thread_count         = -1,
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba  = model.predict_proba(X_val)
        val_pred   = np.argmax(val_proba, axis=1)
        score      = balanced_accuracy_score(y_val, val_pred)
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')

        oof_proba[val_idx] += val_proba
        test_preds         += model.predict_proba(X_test) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_proba, axis=1))
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean fold: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')

    all_oof_proba  += oof_proba  / len(SEEDS)
    all_test_preds += test_preds / len(SEEDS)

# ── Final score ──────────────────────────────────────────────────────────────
final_oof_score = balanced_accuracy_score(y_train, np.argmax(all_oof_proba, axis=1))
print(f"\n{'='*60}")
print(f'FINAL OOF BA (avg over {len(SEEDS)} seeds): {final_oof_score:.4f}')
print(f"{'='*60}")


======================================== SEED=42 ========================================
  Fold  1: BA=0.3937  best_iter=93
  Fold  2: BA=0.4078  best_iter=92
  Fold  3: BA=0.4276  best_iter=66
  Fold  4: BA=0.3708  best_iter=26
  Fold  5: BA=0.4215  best_iter=152
  Fold  6: BA=0.4507  best_iter=193
  Fold  7: BA=0.3560  best_iter=8
  Fold  8: BA=0.3659  best_iter=14
  Fold  9: BA=0.4203  best_iter=89
  Fold 10: BA=0.4043  best_iter=47
  OOF BA (seed=42): 0.4023 | mean fold: 0.4019 ± 0.0287

======================================== SEED=7 ========================================
  Fold  1: BA=0.3790  best_iter=106
  Fold  2: BA=0.4008  best_iter=8
  Fold  3: BA=0.4191  best_iter=86
  Fold  4: BA=0.4065  best_iter=86
  Fold  5: BA=0.3684  best_iter=117
  Fold  6: BA=0.4026  best_iter=86
  Fold  7: BA=0.3963  best_iter=35
  Fold  8: BA=0.3605  best_iter=23
  Fold  9: BA=0.3876  best_iter=90
  Fold 10: BA=0.3843  best_iter=49
  OOF BA (seed=7): 0.3906 | mean fold: 0.3905 ± 0.0171

=====

In [7]:
# ── Cell 7: Per-class recall analysis ───────────────────────────────────────
from sklearn.metrics import classification_report

oof_labels = np.argmax(all_oof_proba, axis=1)
disorder_names = {
    0:'레베르시', 1:'낭포성섬유증', 2:'당뇨', 3:'리증후군', 4:'암',
    5:'테이-삭스', 6:'혈색소침착증', 7:'사립체근병종', 8:'알츠하이머', 9:'확인안됨'
}
report = classification_report(y_train, oof_labels, output_dict=True)
print('Per-class recall:')
for cls in range(10):
    r = report[str(cls)]['recall']
    n = sum(y_train == cls)
    flag = ' ← LOW' if r < 0.3 else ''
    print(f'  Class {cls} ({disorder_names[cls]:12s}): recall={r:.3f}  n={n}{flag}')

Per-class recall:
  Class 0 (레베르시        ): recall=0.288  n=389 ← LOW
  Class 1 (낭포성섬유증      ): recall=0.394  n=2068
  Class 2 (당뇨          ): recall=0.270  n=1090 ← LOW
  Class 3 (리증후군        ): recall=0.358  n=3096
  Class 4 (암           ): recall=0.810  n=58
  Class 5 (테이-삭스       ): recall=0.317  n=1700
  Class 6 (혈색소침착증      ): recall=0.459  n=813
  Class 7 (사립체근병종      ): recall=0.240  n=2643 ← LOW
  Class 8 (알츠하이머       ): recall=0.604  n=91
  Class 9 (확인안됨        ): recall=0.106  n=1301 ← LOW


In [8]:
# ── Cell 8: Save submission ──────────────────────────────────────────────────
final_preds = np.argmax(all_test_preds, axis=1)
submission  = pd.DataFrame({'id': TEST_DATA.index, 'disorder': final_preds}).set_index('id')
submission.to_csv('submission_v2_combined.csv')

print('Saved! submission_v2_combined.csv')
print(f'Shape: {submission.shape}')
print('Prediction distribution:')
print(submission['disorder'].value_counts().sort_index())

Saved! submission_v2_combined.csv
Shape: (8834, 1)
Prediction distribution:
disorder
0     364
1    1433
2     609
3    1662
4     330
5    1249
6    1050
7    1202
8     277
9     658
Name: count, dtype: int64
